# Ingest Observability Data into DOG with OpenTelemetry

Run this notebook from JupyterLab launched at the course root. The `lab.shell(...)` cells use the course root to mount the YAML configuration files. The Doris FE log path is discovered from the running `doris` container.

## 1. Send logs to DOG

Use the following command to write log data to DOG through OTel. It reads the actual FE log directory from the `doris` container's host mount, so it does not depend on the current working directory or the location where `ai-observe-stack` was cloned:

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [4]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


In [5]:
lab.shell(r"""
set -euo pipefail

DORIS_CONTAINER="${DORIS_CONTAINER:-doris}"
CONFIG_DIR="level1/module02-data-ingest"
DORIS_FE_LOG_DIR="$(docker inspect --format '{{range .Mounts}}{{if eq .Destination "/opt/apache-doris/fe/log"}}{{.Source}}{{end}}{{end}}' "$DORIS_CONTAINER")"

if [ -z "$DORIS_FE_LOG_DIR" ]; then
  echo "Could not find the /opt/apache-doris/fe/log mount on the $DORIS_CONTAINER container."
  echo "Run module 1's prepare_environment() cell first, then rerun this cell."
  exit 1
fi

# Docker Desktop on macOS reports host mounts with a /host_mnt prefix.
DORIS_FE_LOG_DIR="${DORIS_FE_LOG_DIR#/host_mnt}"

DORIS_FE_LOG="$DORIS_FE_LOG_DIR/fe.log"
if [ ! -f "$DORIS_FE_LOG" ]; then
  echo "The mounted Doris FE log was not found at $DORIS_FE_LOG"
  exit 1
fi

docker rm -f log2doris >/dev/null 2>&1 || true
docker run --rm -d \
  --name log2doris \
  --network docker_aiobs-net \
  -v "$PWD/$CONFIG_DIR/otel_for_logs.yaml:/etc/otel_for_logs.yaml:ro" \
  -v "$DORIS_FE_LOG:/var/log/fe.log:ro" \
  otel/opentelemetry-collector-contrib:0.144.0 \
  --config /etc/otel_for_logs.yaml

docker inspect --format 'collector={{.Name}} status={{.State.Status}}' log2doris
""", title="Start the Doris log collector")


3f47d23260458ef524b8d1b9befa5a2faba21e88ba1e02e9ce51bfa453681851
collector=/log2doris status=running


CompletedProcess(args=['/bin/bash', '-lc', 'set -euo pipefail\n\nDORIS_CONTAINER="${DORIS_CONTAINER:-doris}"\nCONFIG_DIR="level1/module02-data-ingest"\nDORIS_FE_LOG_DIR="$(docker inspect --format \'{{range .Mounts}}{{if eq .Destination "/opt/apache-doris/fe/log"}}{{.Source}}{{end}}{{end}}\' "$DORIS_CONTAINER")"\n\nif [ -z "$DORIS_FE_LOG_DIR" ]; then\n  echo "Could not find the /opt/apache-doris/fe/log mount on the $DORIS_CONTAINER container."\n  echo "Run module 1\'s prepare_environment() cell first, then rerun this cell."\n  exit 1\nfi\n\n# Docker Desktop on macOS reports host mounts with a /host_mnt prefix.\nDORIS_FE_LOG_DIR="${DORIS_FE_LOG_DIR#/host_mnt}"\n\nDORIS_FE_LOG="$DORIS_FE_LOG_DIR/fe.log"\nif [ ! -f "$DORIS_FE_LOG" ]; then\n  echo "The mounted Doris FE log was not found at $DORIS_FE_LOG"\n  exit 1\nfi\n\ndocker rm -f log2doris >/dev/null 2>&1 || true\ndocker run --rm -d \\\n  --name log2doris \\\n  --network docker_aiobs-net \\\n  -v "$PWD/$CONFIG_DIR/otel_for_logs.yaml:/et

## 2. Send metrics to DOG

Use the following command to write metrics data to DOG through OTel. This example directly uses the metrics exposed by Doris:

In [6]:
lab.shell(r"""
set -euo pipefail

CONFIG_DIR="level1/module02-data-ingest"
docker rm -f metric2doris >/dev/null 2>&1 || true
docker run --rm -d \
  --name metric2doris \
  --network docker_aiobs-net \
  -v "$PWD/$CONFIG_DIR/otel_for_metrics.yaml:/etc/otel_for_metrics.yaml:ro" \
  otel/opentelemetry-collector-contrib:0.144.0 \
  --config /etc/otel_for_metrics.yaml

docker inspect --format 'collector={{.Name}} status={{.State.Status}}' metric2doris
""", title="Start the Doris metric collector")


43648c9031761a723a05fcccb60f63bb8e9f8bdde8ad2837053e618a1428dac6
collector=/metric2doris status=running


CompletedProcess(args=['/bin/bash', '-lc', 'set -euo pipefail\n\nCONFIG_DIR="level1/module02-data-ingest"\ndocker rm -f metric2doris >/dev/null 2>&1 || true\ndocker run --rm -d \\\n  --name metric2doris \\\n  --network docker_aiobs-net \\\n  -v "$PWD/$CONFIG_DIR/otel_for_metrics.yaml:/etc/otel_for_metrics.yaml:ro" \\\n  otel/opentelemetry-collector-contrib:0.144.0 \\\n  --config /etc/otel_for_metrics.yaml\n\ndocker inspect --format \'collector={{.Name}} status={{.State.Status}}\' metric2doris'], returncode=0, stdout='43648c9031761a723a05fcccb60f63bb8e9f8bdde8ad2837053e618a1428dac6\ncollector=/metric2doris status=running\n', stderr='')

## 3. Send traces to DOG

Use the following command to start OTel. It will write trace data to DOG:

In [7]:
lab.shell(r"""
set -euo pipefail

CONFIG_DIR="level1/module02-data-ingest"
docker rm -f traces2doris >/dev/null 2>&1 || true
docker run --rm -d \
  --name traces2doris \
  --network docker_aiobs-net \
  -v "$PWD/$CONFIG_DIR/otel_for_traces.yaml:/etc/otel_for_traces.yaml:ro" \
  otel/opentelemetry-collector-contrib:0.144.0 \
  --config /etc/otel_for_traces.yaml

docker inspect --format 'collector={{.Name}} status={{.State.Status}}' traces2doris
""", title="Start the trace collector")


063923c4df244a2d1362b770fa678f51f0cb00806abf720f7ebeb38729a8df6d
collector=/traces2doris status=running


CompletedProcess(args=['/bin/bash', '-lc', 'set -euo pipefail\n\nCONFIG_DIR="level1/module02-data-ingest"\ndocker rm -f traces2doris >/dev/null 2>&1 || true\ndocker run --rm -d \\\n  --name traces2doris \\\n  --network docker_aiobs-net \\\n  -v "$PWD/$CONFIG_DIR/otel_for_traces.yaml:/etc/otel_for_traces.yaml:ro" \\\n  otel/opentelemetry-collector-contrib:0.144.0 \\\n  --config /etc/otel_for_traces.yaml\n\ndocker inspect --format \'collector={{.Name}} status={{.State.Status}}\' traces2doris'], returncode=0, stdout='063923c4df244a2d1362b770fa678f51f0cb00806abf720f7ebeb38729a8df6d\ncollector=/traces2doris status=running\n', stderr='')

## 4. Simulate trace data

Use the following Python code to simulate sending trace data. First, run the pip command:

In [8]:
%pip install opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-http



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Run the following command to send trace data to OTel. OTel will write the data to DOG:

In [9]:
from opentelemetry import trace
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

resource = Resource.create({"service.name": "python-demo-service"})

provider = TracerProvider(resource=resource)
processor = BatchSpanProcessor(OTLPSpanExporter(endpoint="http://localhost:4318/v1/traces"))
provider.add_span_processor(processor)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("demo-tracer")

with tracer.start_as_current_span("demo-span") as span:
    span.set_attribute("custom.attribute", "hello-otel")
    span.set_attribute("http.status_code", 200)
    print("Data sent successfully!")


Data sent successfully!
